In [1]:
import numpy as np
import pandas as pd
from plyfile import PlyData
import taichi as ti
import os
import taichi.math as fuck

# 初始化Taichi
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
ti.init(arch=ti.cuda)
res = 0.0002
dims = (200, 520, 400)
grid_points = np.prod(dims)
grid_location = np.zeros((grid_points, 3), dtype=int)
grid_intensity = np.zeros((grid_points,), dtype=float)

# 填充grid_location

@ti.kernel
def init_grid_location(
    location:ti.types.ndarray(),
):
    for idx in ti.ndrange(41600000):
        i = (idx%(200*520))%200
        j = (idx%(200*520))/200
        k = idx/(200*520)
        location[idx,0] = i
        location[idx,1] = j
        location[idx,2] = k

init_grid_location(grid_location)
grid_location = grid_location


[Taichi] version 1.7.0, llvm 15.0.4, commit 2fd24490, linux, python 3.10.13


[I 07/02/25 23:59:39.798 1842700] [shell.py:_shell_pop_print@23] Graphical python shell detected, using wrapped sys.stdout


[Taichi] Starting on arch=cuda


[W 07/02/25 23:59:40.484 1842700] 
File "/tmp/ipykernel_1842700/1600269170.py", line 28, in init_grid_location:
        location[idx,1] = j
        ^^^^^^^^^^^^^^^^^^^
Assign may lose precision: i64 <- f32
[W 07/02/25 23:59:40.484 1842700] 
File "/tmp/ipykernel_1842700/1600269170.py", line 29, in init_grid_location:
        location[idx,2] = k
        ^^^^^^^^^^^^^^^^^^^
Assign may lose precision: i64 <- f32


In [ ]:
# 读入点云文件
ply_path = 'iteration_point_cloud_edition62/ball_40_after.ply'
plydata = PlyData.read(ply_path)

# 提取点云数据到相应的数组
cloud_location = np.zeros((plydata['vertex'].count, 3))
source_p0 = np.zeros((plydata['vertex'].count,))
radius = np.zeros((plydata['vertex'].count,))

for i, vertex in enumerate(plydata['vertex']):
    cloud_location[i] = [vertex['x'], vertex['y'], vertex['z']]
    source_p0[i] = vertex['pressure_0']
    radius[i] = vertex['radius']

cloud_num = len(radius)

In [3]:
@ti.func
def softplus(x: float) -> float:
    return ti.log(1.0 + ti.exp(x))

@ti.func
def caculate(a0, p0, xg, yg, zg, xc, yc, zc):
    R = ti.Vector([xg - xc, yg - yc, zg - zc]).norm() 
    intensity = p0 * fuck.step(0.0, a0  - R)
    return intensity

@ti.kernel
def rendering(
    grid_location: ti.types.ndarray(),
    cloud_location: ti.types.ndarray(),
    source_p0: ti.types.ndarray(),
    radius: ti.types.ndarray(),
    grid_intensity: ti.types.ndarray(),
):
    for idx in ti.ndrange(grid_points):
        xg = grid_location[idx, 0]*res
        yg = grid_location[idx, 1]*res
        zg = grid_location[idx, 2]*res
        for num in ti.ndrange(cloud_num):
            xc = cloud_location[num, 0]
            yc = cloud_location[num, 1]
            zc = cloud_location[num, 2]
            p0 = source_p0[num]
            # a0 = radius[num]
            a0 = softplus(radius[num])
            grid_intensity[idx] +=(
                caculate(0.5*a0, 10 * p0 / 55, xg, yg, zg, xc, yc, zc)
                + caculate(0.6*a0, 9 * p0 / 55, xg, yg, zg, xc, yc, zc)
                + caculate(0.9*a0, 8 * p0 / 55, xg, yg, zg, xc, yc, zc)
                + caculate(1.2*a0, 7 * p0 / 55, xg, yg, zg, xc, yc, zc)
                + caculate(1.5*a0, 6 * p0 / 55, xg, yg, zg, xc, yc, zc)
                + caculate(1.8*a0, 5 * p0 / 55, xg, yg, zg, xc, yc, zc)
                + caculate(2.1*a0, 4 * p0 / 55, xg, yg, zg, xc, yc, zc)
                + caculate(2.4*a0, 3 * p0 / 55, xg, yg, zg, xc, yc, zc)
                + caculate(2.7*a0, 2 * p0 / 55, xg, yg, zg, xc, yc, zc)
                + caculate(3.0*a0, 1 * p0 / 55, xg, yg, zg, xc, yc, zc)
            )

In [4]:
import time

start = time.time()
rendering(
    grid_location,
    cloud_location,
    source_p0,
    radius,
    grid_intensity,
)
end = time.time()
print(f"Time cost: {end - start}s")


Time cost: 1121.638026714325s


In [5]:
grid_intensity_to_be_reshape = grid_intensity
print(grid_intensity_to_be_reshape.shape)

(41600000,)


In [6]:
grid_intensity_reshaped = grid_intensity_to_be_reshape.reshape(400,520,200)

In [ ]:
import scipy.io as sio
output_filename = 'Sling_finger_2006.mat'
sio.savemat(output_filename, {'Sling_finger_2006': grid_intensity_reshaped})